# 05 — Retrieval & LLM setup

Assumes previous steps have produced passages, embeddings, and an index.

## Retrieval Step

In [11]:
import os, json, time
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
try:
    import faiss
except ImportError as e:
    raise RuntimeError("faiss is required. Install with: pip install faiss-cpu") from e

# ---- reuse your existing globals if already set in the kernel
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd()))
MODEL_NAME   = globals().get("MODEL_NAME", "intfloat/e5-small-v2")

# Embeddings / index locations you already created
EMB_PATH     = EMB_DIR/"emb.npy"
IDS_PATH     = EMB_DIR/"ids.npy"
META_PATH    = EMB_DIR / "meta.json"

# HNSW index (adjust if your folder name differs)
HNSW_M            = int(os.environ.get("HNSW_M", 32))
HNSW_EF_CONSTRUCT = int(os.environ.get("HNSW_EF_CONSTRUCT", 200))
HNSW_EF_SEARCH    = int(os.environ.get("HNSW_EF_SEARCH", 64))

MODEL_DIRNAME = MODEL_NAME.replace("/", "_")

INDEX_DIR  = PROJECT_ROOT / "indexes" / MODEL_DIRNAME / CFG_NAME / f"hnsw_M{HNSW_M}_efC{HNSW_EF_CONSTRUCT}_efS{HNSW_EF_SEARCH}"
INDEX_PATH = INDEX_DIR / "index.faiss"

# Passages dataframe you already used to build embeddings
CANONICAL_PASSAGES_PATH = EMB_DIR / "passages_canonical.parquet"
PASSAGES_DF = pd.read_parquet(CANONICAL_PASSAGES_PATH)

print("Loaded canonical passages:", CANONICAL_PASSAGES_PATH)
print("Rows:", len(PASSAGES_DF))

if PASSAGES_DF is None:
    raise RuntimeError("PASSAGES_DF must be set to your passages DataFrame with columns: passage_id, doc_id, text")

required_cols = {"passage_id", "doc_id", "text"}
missing = required_cols - set(PASSAGES_DF.columns)
if missing:
    raise KeyError(f"PASSAGES_DF missing columns: {missing}")

# Optional id map (int_id -> original passage_id) if you saved it
ID_MAP_PATH = EMB_DIR / "id_map.csv"
ID_MAP_DF = pd.read_csv(ID_MAP_PATH) if ID_MAP_PATH.exists() else None

# sanity checks
assert (EMB_DIR / "id_map.csv").exists(), "id_map.csv missing!"
assert (EMB_DIR / "passages_canonical.parquet").exists(), "passages_canonical.parquet missing!"

ID_MAP_DF = pd.read_csv(EMB_DIR / "id_map.csv")
assert len(ID_MAP_DF) == len(PASSAGES_DF), f"Mismatch: id_map {len(ID_MAP_DF)} vs passages {len(PASSAGES_DF)}"

Loaded canonical passages: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\embeddings\intfloat_e5-small-v2\chunk_w512_ov0.25\passages_canonical.parquet
Rows: 10938


In [12]:
from typing import Optional, Tuple, List, Dict
from pathlib import Path
import time
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

class DenseRetriever:
    """
    Dense retriever on FAISS HNSW index with timing hooks.
    - Encodes query with the SAME model as passages.
    - Normalizes query embeddings (cosine/IP match).
    - Timed search returns t_embed_q (ms), t_search (ms), t_total_retrieval (ms).
    """

    def __init__(
        self,
        model_name: str,
        index_path: Path,
        passages_df: pd.DataFrame,
        id_map_df: Optional[pd.DataFrame] = None,
        normalize_embeddings: bool = True,
        device: str = "cpu",
        debug: bool = False,
    ):
        self.model_name = model_name
        self.index_path = Path(index_path)
        self.normalize = normalize_embeddings
        self.device = device
        self.debug = debug

        if not self.index_path.exists():
            raise FileNotFoundError(f"Index not found: {self.index_path}")

        # Load FAISS index
        self.index = faiss.read_index(str(self.index_path))

        # Load embedding model
        self.model = SentenceTransformer(model_name, device=device)

        # Store passages (canonical df is recommended)
        self.passages_df = passages_df.copy()
        if "passage_id" not in self.passages_df.columns:
            raise KeyError("passages_df must contain 'passage_id'")
        self.passages_df["passage_id"] = self.passages_df["passage_id"].astype(str)

        # --- DEBUG: check uniqueness assumptions ---
        if self.debug:
            print("\n[DEBUG:init] passages_df rows:", len(self.passages_df))
            print("[DEBUG:init] passages_df passage_id unique?:", self.passages_df["passage_id"].is_unique)
            if "faiss_id" in self.passages_df.columns:
                print("[DEBUG:init] passages_df faiss_id unique?:", self.passages_df["faiss_id"].is_unique)

        # Build mapping from FAISS id -> passage_id
        self.int_to_pid = None
        if id_map_df is not None:
            cols = set(id_map_df.columns)
            if {"faiss_id", "passage_id"} <= cols:
                if self.debug:
                    print("\n[DEBUG:init] id_map_df rows:", len(id_map_df))
                    print("[DEBUG:init] id_map_df faiss_id unique?:", id_map_df["faiss_id"].is_unique)
                    print("[DEBUG:init] id_map_df passage_id unique?:", id_map_df["passage_id"].is_unique)

                self.int_to_pid = dict(
                    zip(
                        id_map_df["faiss_id"].astype(np.int64),
                        id_map_df["passage_id"].astype(str),
                    )
                )

                if self.debug:
                    print("[DEBUG:init] int_to_pid size:", len(self.int_to_pid))

            elif {"int_id", "passage_id"} <= cols:
                self.int_to_pid = dict(
                    zip(
                        id_map_df["int_id"].astype(np.int64),
                        id_map_df["passage_id"].astype(str),
                    )
                )

                if self.debug:
                    print("\n[DEBUG:init] (legacy) int_to_pid size:", len(self.int_to_pid))

    def set_ef_search(self, ef: int):
        """Set efSearch on underlying HNSW index (handles IndexIDMap2 wrapper)."""
        inner = self.index
        if isinstance(inner, (faiss.IndexIDMap, faiss.IndexIDMap2)):
            inner = inner.index
        if hasattr(inner, "hnsw"):
            inner.hnsw.efSearch = int(ef)

    def _encode_query(self, q: str) -> Tuple[np.ndarray, float]:
        t0 = time.perf_counter()
        q_emb = self.model.encode(
            [f"query: {q}"],
            convert_to_numpy=True,
            normalize_embeddings=self.normalize,
            show_progress_bar=False,
        ).astype("float32", copy=False)
        t_ms = (time.perf_counter() - t0) * 1000.0
        return q_emb, t_ms

    def _ids_to_rows(self, ids: List[int]) -> pd.DataFrame:
        """
        Convert FAISS integer ids into rows of passages_df.
        Preserves order and length (left-join), so downstream alignment stays safe.
        """
        if self.int_to_pid is not None:
            pids = [self.int_to_pid.get(int(i)) for i in ids]

            if self.debug:
                # how many mapped vs missing?
                n_none = sum(x is None for x in pids)
                print("\n[DEBUG:_ids_to_rows] len(ids):", len(ids))
                print("[DEBUG:_ids_to_rows] len(pids):", len(pids), "missing passage_id:", n_none)
                # duplicates in pids can cause merge expansion if passages_df has duplicate passage_id
                print("[DEBUG:_ids_to_rows] unique pids (incl None):", len(set(pids)))

            ordered = pd.DataFrame({"passage_id": pd.Series(pids, dtype="string")})

            # DEBUG: check if passages_df has duplicate passage_id
            if self.debug:
                dup_pid = self.passages_df["passage_id"].duplicated().sum()
                print("[DEBUG:_ids_to_rows] passages_df duplicate passage_id count:", int(dup_pid))

            if self.debug:
                print("[DEBUG:_ids_to_rows] passages_df passage_id unique?:", self.passages_df["passage_id"].is_unique)
                print("[DEBUG:_ids_to_rows] duplicate passage_id count:", int(self.passages_df["passage_id"].duplicated().sum()))

            merged = ordered.merge(
                self.passages_df,
                on="passage_id",
                how="left",
                sort=False,
            )
            
            if len(merged) != len(ordered):
                raise ValueError(
                    "Mapping expanded (passage_id not unique or wrong passages/id_map loaded). "
                    f"ordered={len(ordered)} merged={len(merged)}"
                    )


            if self.debug:
                print("[DEBUG:_ids_to_rows] len(ordered):", len(ordered), "len(merged):", len(merged))
                if len(merged) != len(ordered):
                    # This is the smoking gun: one-to-many merge expansion.
                    vc = merged["passage_id"].value_counts(dropna=False).head(5)
                    print("[DEBUG:_ids_to_rows] TOP expanded passage_ids (value_counts):\n", vc)

            return merged

        # Fallback
        return self.passages_df.iloc[ids].copy()

    def timed_search(
        self,
        query: str,
        k: int = 10,
        efsearch: Optional[int] = None,
    ) -> Tuple[pd.DataFrame, Dict[str, float]]:

        if efsearch is not None:
            self.set_ef_search(efsearch)

        # 1) Encode query
        q_emb, t_embed_q = self._encode_query(query)

        # 2) FAISS search
        t0 = time.perf_counter()
        scores, idx = self.index.search(q_emb, int(k))
        t_search = (time.perf_counter() - t0) * 1000.0

        # Flatten
        idx = np.asarray(idx).reshape(-1)
        scores = np.asarray(scores).reshape(-1)

        if self.debug:
            print("\n[DEBUG:timed_search] k:", k)
            print("[DEBUG:timed_search] raw idx shape:", idx.shape, "raw scores shape:", scores.shape)
            print("[DEBUG:timed_search] raw idx[:10]:", idx[:10].tolist())

        # Filter invalid ids
        valid = idx != -1
        idx = idx[valid]
        scores = scores[valid]

        ids = idx.tolist()
        scr = scores.tolist()

        if self.debug:
            print("[DEBUG:timed_search] after valid filter len(ids):", len(ids), "len(scr):", len(scr))
            print("[DEBUG:timed_search] unique ids:", len(set(ids)))

        # 3) Map ids -> rows
        rows = self._ids_to_rows(ids)

        if self.debug:
            print("[DEBUG:timed_search] after _ids_to_rows len(rows):", len(rows), "len(scr):", len(scr))
            
            # if len(rows) > len(scr) then merge expanded rows OR something duplicated
            if len(rows) != len(scr):
                print("[DEBUG:timed_search] LENGTH MISMATCH detected right after mapping!")

        # If mapping failed for some rows, drop them and drop matching scores
        if "text" in rows.columns:
            if self.debug:
                print("[DEBUG:timed_search] before text notna filter len(rows):", len(rows), "len(scr):", len(scr))
                print("[DEBUG:timed_search] rows['text'] nulls:", int(rows["text"].isna().sum()))

            ok = rows["text"].notna().to_numpy()
            rows = rows.loc[ok].copy()
            # NOTE: this is where your crash happens if ok length != len(scr)
            scr = list(np.asarray(scr)[ok])

            if self.debug:
                print("[DEBUG:timed_search] after text notna filter len(rows):", len(rows), "len(scr):", len(scr))

        # Final safety alignment
        n = min(len(rows), len(scr))
        rows = rows.iloc[:n].copy()
        scr = scr[:n]

        rows["score"] = scr
        rows["rank"] = np.arange(1, n + 1)

        for col in ["doc_id", "text"]:
            if col not in rows.columns:
                rows[col] = None

        df = rows[["rank", "score", "passage_id", "doc_id", "text"]]

        timers = {
            "t_embed_q": float(t_embed_q),
            "t_search": float(t_search),
            "t_total_retrieval": float(t_embed_q + t_search),
        }
        return df, timers


In [13]:
from typing import Optional, Tuple, Dict, List
import numpy as np
import pandas as pd
import time

from sklearn.feature_extraction.text import TfidfVectorizer


class HybridRetriever:
    """
    Hybrid retriever = dense (FAISS) + lexical (TF-IDF).

    - Uses an existing DenseRetriever instance.
    - Builds a TF-IDF index over passages_df[text_col].
    - At query time, gets candidates from both,
      min-max normalizes scores, and linearly combines them.

    alpha = weight for dense scores (0..1).
            (1-alpha) is weight for lexical scores.
    """

    def __init__(
        self,
        dense_retriever,               # DenseRetriever
        passages_df: pd.DataFrame,
        alpha: float = 0.5,
        text_col: str = "text",
        tfidf_stop_words: str = "english",
        tfidf_max_df: float = 0.95,
    ):
        self.dense = dense_retriever
        self.alpha = float(alpha)
        if not (0.0 <= self.alpha <= 1.0):
            raise ValueError("alpha must be between 0 and 1")

        if "passage_id" not in passages_df.columns:
            raise KeyError("passages_df must contain 'passage_id'")
        if text_col not in passages_df.columns:
            raise KeyError(f"passages_df must contain '{text_col}'")

        # Keep copy (do NOT reorder unless you mean to)
        self.passages_df = passages_df.copy()
        self.passages_df["passage_id"] = self.passages_df["passage_id"].astype(str)

        # Ensure required output columns exist
        for col in ["doc_id", "text"]:
            if col not in self.passages_df.columns:
                self.passages_df[col] = None

        # Map passage_id -> row index (fast lookups)
        # If passage_id has duplicates, last one wins (should not happen in your canonical pipeline).
        self.pid2row = {pid: i for i, pid in enumerate(self.passages_df["passage_id"].tolist())}

        # ---- Build TF-IDF index over passage text ----
        texts = self.passages_df[text_col].fillna("").astype(str).tolist()
        self.vectorizer = TfidfVectorizer(stop_words=tfidf_stop_words, max_df=tfidf_max_df)
        self.tfidf = self.vectorizer.fit_transform(texts)  # shape: [N, V]

    def set_ef_search(self, ef_search: int):
        """Only dense FAISS uses efSearch."""
        self.dense.set_ef_search(ef_search)

    @staticmethod
    def _minmax(x: np.ndarray) -> np.ndarray:
        if x.size == 0:
            return x
        xmin = float(x.min())
        xmax = float(x.max())
        if xmax <= xmin:
            return np.zeros_like(x, dtype=np.float32)
        return ((x - xmin) / (xmax - xmin)).astype(np.float32)

    def timed_search(
        self,
        query: str,
        k: int = 10,
        efsearch: Optional[int] = None,
        candidate_mult: int = 4,
    ) -> Tuple[pd.DataFrame, Dict[str, float]]:
        """
        Return a DataFrame with columns:
          [rank, score, passage_id, doc_id, text]
        where score is the HYBRID (dense + lexical) score.

        Timers dict has:
          t_embed_q, t_search, t_total_retrieval (ms)
        """
        t0 = time.perf_counter()

        # ---------- 1) DENSE SIDE ----------
        dense_k = max(k * candidate_mult, k)
        dense_df, dense_timers = self.dense.timed_search(
            query,
            k=dense_k,
            efsearch=efsearch,
        )

        # Dense results might be empty if index/search fails
        dense_pids = dense_df["passage_id"].astype(str).tolist() if len(dense_df) else []
        dense_scores_list = dense_df["score"].tolist() if len(dense_df) else []
        dense_score_map = dict(zip(dense_pids, dense_scores_list))

        # ---------- 2) LEXICAL SIDE (TF-IDF) ----------
        t_lex_start = time.perf_counter()

        q_vec = self.vectorizer.transform([query])          # [1, V]
        scores_lex = (self.tfidf @ q_vec.T).toarray().ravel()  # [N]

        lex_k = min(max(k * candidate_mult, k), scores_lex.shape[0])
        if lex_k > 0:
            lex_idx = np.argpartition(-scores_lex, lex_k - 1)[:lex_k]
        else:
            lex_idx = np.array([], dtype=int)

        lex_pids = self.passages_df.iloc[lex_idx]["passage_id"].astype(str).tolist() if lex_k > 0 else []

        t_lex_end = time.perf_counter()

        # Map lexical scores
        lex_score_map = {pid: float(scores_lex[self.pid2row[pid]]) for pid in lex_pids}

        # ---------- 3) FUSION ----------
        # Union of dense + lexical candidates (preserve order)
        candidate_pids = list(dict.fromkeys(dense_pids + lex_pids))

        if not candidate_pids:
            # return empty result with timers
            empty = pd.DataFrame(columns=["rank", "score", "passage_id", "doc_id", "text"])
            t_total_ms = (time.perf_counter() - t0) * 1000.0
            timers = {
                "t_embed_q": float(dense_timers.get("t_embed_q", 0.0)) if isinstance(dense_timers, dict) else 0.0,
                "t_search": max(t_total_ms, 0.0),
                "t_total_retrieval": t_total_ms,
            }
            return empty, timers

        dense_scores = np.array([dense_score_map.get(pid, 0.0) for pid in candidate_pids], dtype=np.float32)
        lex_scores = np.array([lex_score_map.get(pid, 0.0) for pid in candidate_pids], dtype=np.float32)

        dense_norm = self._minmax(dense_scores)
        lex_norm = self._minmax(lex_scores)

        hybrid_scores = self.alpha * dense_norm + (1.0 - self.alpha) * lex_norm

        # Top-k
        k_eff = min(int(k), len(candidate_pids))
        order = np.argsort(-hybrid_scores)[:k_eff]
        final_pids = [candidate_pids[i] for i in order]
        final_scores = hybrid_scores[order]

        # Lookup rows
        final_idx = [self.pid2row[pid] for pid in final_pids]
        rows = self.passages_df.iloc[final_idx].copy()
        rows["score"] = final_scores.astype(np.float32)
        rows["rank"] = np.arange(1, len(rows) + 1)

        df = rows[["rank", "score", "passage_id", "doc_id", "text"]]

        # ---------- Timers ----------
        t_total_ms = (time.perf_counter() - t0) * 1000.0
        t_embed_q = float(dense_timers.get("t_embed_q", 0.0))
        t_search = max(t_total_ms - t_embed_q, 0.0)

        timers = {
            "t_embed_q": t_embed_q,
            "t_search": t_search,
            "t_total_retrieval": t_total_ms,
            "t_lexical_ms": (t_lex_end - t_lex_start) * 1000.0,  # extra (optional)
        }

        return df, timers


In [14]:
from typing import Optional, Tuple, Dict
import numpy as np
import pandas as pd
import time

from sklearn.feature_extraction.text import CountVectorizer


class BM25Retriever:
    """
    Pure lexical BM25 retriever over passages_df['text'].

    - Uses CountVectorizer to build a term-count matrix.
    - Precomputes BM25 stats (idf, doc lengths, avgdl).
    - Exposes the SAME interface as DenseRetriever / HybridRetriever:
        timed_search(query, k, efsearch=None) -> (df, timers)
    """

    def __init__(
        self,
        passages_df: pd.DataFrame,
        k1: float = 1.5,
        b: float = 0.75,
        text_col: str = "text",
    ):
        if "passage_id" not in passages_df.columns:
            raise KeyError("passages_df must contain 'passage_id'")
        if text_col not in passages_df.columns:
            raise KeyError(f"passages_df must contain '{text_col}'")

        self.passages_df = passages_df.copy()

        # Ensure required output columns exist (so evaluation doesn't crash)
        self.passages_df["passage_id"] = self.passages_df["passage_id"].astype(str)
        if "doc_id" not in self.passages_df.columns:
            self.passages_df["doc_id"] = None
        if "text" not in self.passages_df.columns:
            self.passages_df["text"] = self.passages_df[text_col]

        self.k1 = float(k1)
        self.b = float(b)
        self.text_col = text_col

        # 1) Build term-count matrix
        texts = self.passages_df[self.text_col].fillna("").astype(str).tolist()
        self.vectorizer = CountVectorizer()
        X = self.vectorizer.fit_transform(texts)  # sparse (n_docs, n_terms)
        self.X = X

        # 2) Precompute BM25 stats
        self.N = int(X.shape[0])

        # document frequencies (df) per term
        df_t = (X > 0).sum(axis=0).A1.astype(np.float32)  # (n_terms,)
        # BM25 idf
        self.idf = np.log(1.0 + (self.N - df_t + 0.5) / (df_t + 0.5)).astype(np.float32)

        # document lengths and average length
        self.doc_len = X.sum(axis=1).A1.astype(np.float32)  # (n_docs,)
        self.avgdl = float(np.mean(self.doc_len)) if self.N > 0 else 0.0

    def _score(self, query: str) -> np.ndarray:
        """
        Compute BM25 scores for all documents for a single query.
        Returns float32 array of shape (n_docs,).
        """
        if self.N == 0:
            return np.zeros(0, dtype=np.float32)

        q_vec = self.vectorizer.transform([query])  # (1, n_terms)
        q_indices = q_vec.indices  # non-zero term indices in query

        if q_indices.size == 0:
            return np.zeros(self.N, dtype=np.float32)

        # Sparse slice: (n_docs, n_query_terms)
        X_sub = self.X[:, q_indices]

        # Convert only this slice to dense (ok for 10k passages; safer than densifying full X)
        tf = X_sub.toarray().astype(np.float32)  # (n_docs, n_query_terms)

        dl = self.doc_len[:, None]  # (n_docs, 1)
        idf = self.idf[q_indices][None, :]  # (1, n_query_terms)

        k1 = self.k1
        b = self.b
        avgdl = self.avgdl if self.avgdl > 0 else 1.0

        denom = tf + k1 * (1.0 - b + b * dl / avgdl)
        denom = np.maximum(denom, 1e-9)

        num = tf * (k1 + 1.0)
        scores = (idf * (num / denom)).sum(axis=1)  # (n_docs,)

        return scores.astype(np.float32)

    def timed_search(
        self,
        query: str,
        k: int = 10,
        efsearch: Optional[int] = None,  # ignored (dense-only)
    ) -> Tuple[pd.DataFrame, Dict[str, float]]:
        """
        Same interface as DenseRetriever / HybridRetriever:
          - Ignores efsearch.
          - Returns hits dataframe with [rank, score, passage_id, doc_id, text].
          - Returns timers dict: t_embed_q, t_search, t_total_retrieval (ms).
        """
        t0 = time.perf_counter()

        scores = self._score(query)  # (n_docs,)
        n_docs = scores.shape[0]

        k_eff = min(int(k), n_docs)
        if k_eff <= 0:
            empty_df = pd.DataFrame(columns=["rank", "score", "passage_id", "doc_id", "text"])
            t_total_ms = (time.perf_counter() - t0) * 1000.0
            timers = {"t_embed_q": 0.0, "t_search": t_total_ms, "t_total_retrieval": t_total_ms}
            return empty_df, timers

        # Top-k indices
        topk_idx = np.argpartition(-scores, k_eff - 1)[:k_eff]
        topk_idx = topk_idx[np.argsort(-scores[topk_idx])]

        rows = self.passages_df.iloc[topk_idx].copy()
        rows["score"] = scores[topk_idx]
        rows["rank"] = np.arange(1, len(rows) + 1)

        df = rows[["rank", "score", "passage_id", "doc_id", "text"]]

        t_total_ms = (time.perf_counter() - t0) * 1000.0
        timers = {"t_embed_q": 0.0, "t_search": t_total_ms, "t_total_retrieval": t_total_ms}
        return df, timers


In [15]:
import re
from typing import List, Optional, Tuple, Dict

import numpy as np
import pandas as pd
import tracemalloc


def _normalize_text(s: str) -> str:
    s = str(s)  # handle None/NaN
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def _has_answer(passage: str, answers: List[str]) -> bool:
    # passage can be NaN/None if mapping failed; normalize safely
    if passage is None or (isinstance(passage, float) and np.isnan(passage)):
        return False
    p = _normalize_text(passage)
    for a in answers:
        if a is None or (isinstance(a, float) and np.isnan(a)):
            continue
        if _normalize_text(a) in p:
            return True
    return False


def evaluate_retriever(
    retriever,
    queries_df: pd.DataFrame,
    q_col: str,
    supervision: str,                   # "gold_id" or "answers"
    gold_col: Optional[str] = None,     # when supervision="gold_id"
    answers_col: Optional[str] = None,  # when supervision="answers"
    k_list: List[int] = [1, 3, 5, 10],
    efsearch: Optional[int] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs retrieval over queries_df with a given retriever and returns:
      - per_query_df: one row per query with hits@k, latency, memory, and top-k info
      - summary_df:   aggregate Recall@k, MRR@k, and per-K latency/memory metrics

    retriever must implement:
      timed_search(query, k, efsearch=None) -> (df, timers)
    """
    assert supervision in {"gold_id", "answers"}

    if supervision == "gold_id" and not gold_col:
        raise ValueError("gold_col must be provided when supervision='gold_id'")
    if supervision == "answers" and not answers_col:
        raise ValueError("answers_col must be provided when supervision='answers'")

    # Optionally set efSearch once (if retriever supports it)
    if efsearch is not None and hasattr(retriever, "set_ef_search"):
        retriever.set_ef_search(efsearch)

    per_rows = []
    rr_lists = {f"MRR@{k}": [] for k in k_list}
    recall_counts = {f"Recall@{k}": 0 for k in k_list}

    tracemalloc.start()

    max_k = int(max(k_list))
    print("efsearch = ", efsearch)

    for i, row in queries_df.iterrows():
        question = row[q_col]

        tracemalloc.reset_peak()

        # --- 1) Run retrieval ---
        df, timers = retriever.timed_search(question, k=max_k, efsearch=efsearch)

        # Ensure df has expected columns even if empty
        if df is None or len(df) == 0:
            ranked_pids = []
            ranked_texts = []
            ranked_scores = []
        else:
            # Normalize types
            ranked_pids = df.get("passage_id", pd.Series([], dtype=str)).astype(str).tolist()
            ranked_texts = df.get("text", pd.Series([], dtype=object)).tolist()
            ranked_scores = df.get("score", pd.Series([], dtype=float)).tolist()

        # --- 2) Peak memory (MB) ---
        _, peak_bytes = tracemalloc.get_traced_memory()
        mem_peak_mb = peak_bytes / (1024 * 1024)

        # --- 3) Latency (ms) ---
        retrieval_latency_ms = timers.get("t_total_retrieval", timers.get("t_search", np.nan))

        # --- 4) Compute relevance ---
        if supervision == "gold_id":
            gold_pid = str(row[gold_col])

            for k in k_list:
                recall_counts[f"Recall@{k}"] += int(gold_pid in ranked_pids[:k])

            rr = 0.0
            found_rank = None
            for r_idx, pid in enumerate(ranked_pids, start=1):
                if pid == gold_pid:
                    rr = 1.0 / r_idx
                    found_rank = r_idx
                    break

            for k in k_list:
                rr_lists[f"MRR@{k}"].append(rr if (found_rank is not None and found_rank <= k) else 0.0)

            hit_flags = {f"hit@{k}": int(gold_pid in ranked_pids[:k]) for k in k_list}

        else:  # supervision == "answers"
            answers = row[answers_col]

            # Normalize answers into a list[str]
            if answers is None or (isinstance(answers, float) and np.isnan(answers)):
                answers = []
            elif isinstance(answers, str):
                answers = [answers]
            elif not isinstance(answers, (list, tuple, np.ndarray)):
                answers = [str(answers)]
            else:
                answers = list(answers)

            for k in k_list:
                topk_texts = ranked_texts[:k]
                hit = any(_has_answer(t, answers) for t in topk_texts)
                recall_counts[f"Recall@{k}"] += int(hit)

            rr = 0.0
            found_rank = None
            for r_idx, ttxt in enumerate(ranked_texts, start=1):
                if _has_answer(ttxt, answers):
                    rr = 1.0 / r_idx
                    found_rank = r_idx
                    break

            for k in k_list:
                rr_lists[f"MRR@{k}"].append(rr if (found_rank is not None and found_rank <= k) else 0.0)

            hit_flags = {f"hit@{k}": int(any(_has_answer(t, answers) for t in ranked_texts[:k])) for k in k_list}

        # --- 5) Store per-query record ---
        per_rows.append({
            "q_idx": i,
            "question": question,
            **hit_flags,
            "retrieval_latency_ms": retrieval_latency_ms,
            "mem_peak_mb": mem_peak_mb,
            "topk_pids": ranked_pids,
            "topk_scores": ranked_scores,
            "topk_texts": ranked_texts,
        })

    tracemalloc.stop()

    per_query_df = pd.DataFrame(per_rows)
    n = len(per_query_df)

    # --- 6) Aggregate metrics per K ---
    summary: Dict[str, object] = {}

    for k in k_list:
        summary[f"Recall@{k}"] = recall_counts[f"Recall@{k}"] / n if n > 0 else 0.0
        summary[f"MRR@{k}"] = float(np.mean(rr_lists[f"MRR@{k}"])) if rr_lists[f"MRR@{k}"] else 0.0

        # Latency/memory across ALL queries
        summary[f"mean_retrieval_latency_ms@{k}"] = float(np.nanmean(per_query_df["retrieval_latency_ms"])) if n > 0 else np.nan
        summary[f"mean_mem_peak_mb@{k}"] = float(np.nanmean(per_query_df["mem_peak_mb"])) if n > 0 else np.nan
        

        
    summary_df = pd.DataFrame([summary])
    return per_query_df, summary_df


In [16]:
import pandas as pd
from pathlib import Path

# --- Load canonical artifacts (must match the index build) ---
PASSAGES_DF = pd.read_parquet(EMB_DIR / "passages_canonical.parquet")
ID_MAP_DF   = pd.read_csv(EMB_DIR / "id_map.csv")

# --- Make sure INDEX_PATH points to the same place used in the index builder ---
MODEL_DIRNAME = MODEL_NAME.replace("/", "_")
INDEX_DIR = PROJECT_ROOT / "indexes" / MODEL_DIRNAME / CFG_NAME / f"hnsw_M{HNSW_M}_efC{HNSW_EF_CONSTRUCT}_efS{HNSW_EF_SEARCH}"
INDEX_PATH = INDEX_DIR / "index.faiss"

# 1) Dense retriever
dense_retriever = DenseRetriever(
    model_name=MODEL_NAME,
    index_path=INDEX_PATH,
    passages_df=PASSAGES_DF,
    id_map_df=ID_MAP_DF,
    normalize_embeddings=True,
    device="cpu",
    debug=False
)

#dense_retriever.set_ef_search(96)  # good default for evaluation

# 2) Hybrid retriever wrapping the dense one
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    passages_df=PASSAGES_DF,
    alpha=0.5
)

# 3) Pure BM25 retriever (lexical only)
bm25_retriever = BM25Retriever(
    passages_df=PASSAGES_DF,
    k1=1.5,
    b=0.75
)

print("Loaded passages:", len(PASSAGES_DF))
print("Loaded id_map:", len(ID_MAP_DF))
print("Index path:", INDEX_PATH)


Loaded passages: 10938
Loaded id_map: 10938
Index path: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\indexes\intfloat_e5-small-v2\chunk_w512_ov0.25\hnsw_M32_efC200_efS64\index.faiss


In [17]:
# Example: dev_ans_df = pd.read_parquet(PROJECT_ROOT/"data"/"questions_eval.parquet")
# Expect columns: ["question", "answers"] where answers is a string or list[str]

queries_df = pd.read_parquet(Q_PATH)  # same Q_PATH you used when saving df_q_eval

queries_df = queries_df.rename(columns={
    "question_clean": "question"
})

queries_df["answers"] = queries_df["answer_text"].fillna("").apply(
    lambda s: [a.strip() for a in s.split(" , ") if a.strip()]
)

K_LIST = [5,10,50]

per_q_bm25, summary_bm25 = evaluate_retriever(
    retriever=bm25_retriever,
    queries_df=queries_df,
    q_col="question",
    supervision="answers",
    answers_col="answers",
    k_list=K_LIST,
    efsearch=None
)

print("=== BM25 retriever metrics ===")
display(summary_bm25)



# 1) Dense-only metrics
per_q_dense, summary_dense = evaluate_retriever(
    retriever=dense_retriever,
    queries_df=queries_df,
    q_col="question",
    supervision="answers",
    answers_col="answers",
    k_list=K_LIST,
    efsearch=96
)

print("=== Dense retriever metrics ===")
display(summary_dense)

#  Hybrid metrics
per_q_hybrid, summary_hybrid = evaluate_retriever(
    retriever=hybrid_retriever,
    queries_df=queries_df,
    q_col="question",
    supervision="answers",
    answers_col="answers",
    k_list=K_LIST,
    efsearch=96
)

print("=== Hybrid retriever metrics ===")
display(summary_hybrid)



efsearch =  None
=== BM25 retriever metrics ===


,Recall@5,MRR@5,mean_retrieval_latency_ms@5,mean_mem_peak_mb@5,max_mem_peak_mb@5,mean_latency_on_hits_ms@5,Recall@10,MRR@10,mean_retrieval_latency_ms@10,mean_mem_peak_mb@10,max_mem_peak_mb@10,mean_latency_on_hits_ms@10,Recall@50,MRR@50,mean_retrieval_latency_ms@50,mean_mem_peak_mb@50,max_mem_peak_mb@50,mean_latency_on_hits_ms@50
0,0.842105,0.749089,18.216245,4.254257,10.56933,18.206172,0.862348,0.751838,18.216245,4.254257,10.56933,18.221751,0.917004,0.754672,18.216245,4.254257,10.56933,18.218537


efsearch =  96
=== Dense retriever metrics ===


,Recall@5,MRR@5,mean_retrieval_latency_ms@5,mean_mem_peak_mb@5,max_mem_peak_mb@5,mean_latency_on_hits_ms@5,Recall@10,MRR@10,mean_retrieval_latency_ms@10,mean_mem_peak_mb@10,max_mem_peak_mb@10,mean_latency_on_hits_ms@10,Recall@50,MRR@50,mean_retrieval_latency_ms@50,mean_mem_peak_mb@50,max_mem_peak_mb@50,mean_latency_on_hits_ms@50
0,0.744939,0.611404,86.185977,1.792531,2.645985,80.847958,0.803644,0.619628,86.185977,1.792531,2.645985,84.042059,0.884615,0.62442,86.185977,1.792531,2.645985,86.581278


efsearch =  96
=== Hybrid retriever metrics ===


,Recall@5,MRR@5,mean_retrieval_latency_ms@5,mean_mem_peak_mb@5,max_mem_peak_mb@5,mean_latency_on_hits_ms@5,Recall@10,MRR@10,mean_retrieval_latency_ms@10,mean_mem_peak_mb@10,max_mem_peak_mb@10,mean_latency_on_hits_ms@10,Recall@50,MRR@50,mean_retrieval_latency_ms@50,mean_mem_peak_mb@50,max_mem_peak_mb@50,mean_latency_on_hits_ms@50
0,0.785425,0.65614,113.589897,1.833449,2.706285,111.928899,0.850202,0.665562,113.589897,1.833449,2.706285,112.954812,0.906883,0.668886,113.589897,1.833449,2.706285,113.30589


In [18]:
import pandas as pd
from pathlib import Path

# Path to your evaluation questions (same as you used for retrieval metrics)
Q_PATH = PROJECT_ROOT / "data" / "processed" / "questions_eval.parquet"

questions_df = pd.read_parquet(Q_PATH)

# Make sure we have a 'question' column
if "question" not in questions_df.columns and "question_clean" in questions_df.columns:
    questions_df = questions_df.rename(columns={"question_clean": "question"})

print("Total evaluation questions:", len(questions_df))

# (Optional) work on a subset while you debug / run on CPU
N_SAMPLE = 50   # or None for all
if N_SAMPLE is not None:
    questions_df = questions_df.sample(N_SAMPLE, random_state=42).reset_index(drop=True)



Total evaluation questions: 494


,question_id,question,answer_text
0,sfq_26014,The Gulf of Carpentaria is an inlet on the north coast of which country?,"AUSTRALIA , AUSTRALIAN , Asutralia , Ausrtalia , Aussieland , Austalia , Austraila , Austrailia , Australai , AustraliA , Australia , Australia (Commonwealth realm) , Australia (Commonwealth) , Australia (commonwealth realm) , Australia (commonwealth) , Australia (constitutional monarchy) , Australia (country) , Australia (dominion) , Australia (empire) , Australia (federation) , Australia (monarchy) , Australia (nation state) , Australia (nation) , Australia (nation-state) , Australia (realm) , Australia (state) , Australia's , Australia. , Australian Commonwealth , Australian Woman's Day , Australian city life , Australian country life , Australian geopolitics , Australian mainland , Australian's , Australias , Australie , Australien , Australiia , Australija , Australlia , Australo , Australo- , Australocentric , Australocentrism , Australocentrist , Austraya , Austrlaia , Austrlia , City life in Australia , Commonwealth Australia , Commonwealth of Australia , Commonwealth of australia , Continental Australia , Country life in Australia , Dominion of Australia , Empire of Australia , Etymology of Australia , Federal Australia , Geopolitics of Australia , ISO 3166-1:AU , Imperial Australia , Mainland Australia , Modern Australia , New Australian , Orstraya , Ostralia , Pax Australiana , Peace of Australia , Philosophy in Australia , Science in Australia , Straya , Technology in Australia , The Commonwealth of Australia , United States of Australia"


In [3]:
import os
from llama_cpp import Llama

GEN_MODEL_DIR = r"C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\models"

# Use the actual GGUF filename from os.listdir output
MODEL_PATH = os.path.join(
    GEN_MODEL_DIR,
    "Qwen2.5-7B-Instruct-Q4_K_M.gguf",  
)

N_CTX = 2048      # or higher if you want
N_THREADS = 8    # or os.cpu_count()

qwen_llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_threads=N_THREADS,
    verbose=False,
)

print(f"Loaded LLama model from: {MODEL_PATH}")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Loaded LLama model from: C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\models\Qwen2.5-7B-Instruct-Q4_K_M.gguf
